# Microsoft Foundry module level alignment

This notebook compares each module's existing catalog level with a first-pass level derived from the repository's L100-L500 proficiency rubric.

In [1]:
import csv
import re
from collections import Counter
from pathlib import Path

csv_candidates = [
    Path("content-data/microsoft-foundry-modules.csv"),
    Path("../content-data/microsoft-foundry-modules.csv"),
]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find content-data/microsoft-foundry-modules.csv")

with csv_path.open(newline="", encoding="utf-8-sig") as csv_file:
    modules = list(csv.DictReader(csv_file))

## Existing catalog levels

In [2]:
catalog_level_order = ("beginner", "intermediate", "advanced")
catalog_counts = Counter(module["levels"].strip().lower() for module in modules)
catalog_summary = [
    {"catalog_level": level.title(), "module_count": catalog_counts[level]}
    for level in catalog_level_order
]
catalog_summary

[{'catalog_level': 'Beginner', 'module_count': 65},
 {'catalog_level': 'Intermediate', 'module_count': 69},
 {'catalog_level': 'Advanced', 'module_count': 21}]

## Rubric-based tagging

| Rubric level | Focus | Basic definition |
| --- | --- | --- |
| L100: Beginner | Understand and identify | Demonstrates basic awareness of a concept. |
| L200: Intermediate | Apply and describe | Can explain the concept clearly, but stays at a general level. |
| L300: Proficient | Analyze and integrate | Demonstrates practical understanding by reasoning through how the concept applies to a specific context or problem. |
| L400: Advanced | Evaluate and extend | Frames tradeoffs or implications of different approaches using the concept. |
| L500: Frontier Expert | Originate and scale | Demonstrates expert judgment by guiding community thinking, reframing broader problems, and recommending scalable paths forward. |

The source CSV contains titles and metadata, but not the full module text or learning objectives. The deterministic tagging below therefore uses observable action signals in each title. When several signals occur, it assigns the highest rubric level. Titles without a listed signal default to L200 because they describe an applied learning activity. These are reproducible first-pass tags, not validated human ratings.

In [3]:
rubric_level_order = ("L100", "L200", "L300", "L400", "L500")
rubric_rank = {level: rank for rank, level in enumerate(rubric_level_order, start=1)}

rubric_signals = {
    "L100": (
        r"\bintroduction\b", r"\bget started\b", r"\bexplor(?:e|ing)\b",
        r"\bdiscover\b", r"\bunderstand\b", r"\bguide to\b",
        r"\bconcepts\b", r"\bbasics\b",
    ),
    "L200": (
        r"\bapply\b", r"\bconfigure\b", r"\bcreate\b", r"\bbuild\b",
        r"\bdevelop\b", r"\buse\b", r"\bdeploy\b", r"\brun\b",
        r"\bset up\b", r"\bgenerate\b", r"\bextract\b", r"\btranslate\b",
        r"\bsummarize\b", r"\bconnect\b", r"\bconvert\b", r"\bmanage\b",
        r"\bgovern\b", r"\bsecure\b", r"\bprotect\b", r"\bspecify\b",
        r"\baccelerate\b", r"\badd\b", r"\benhance\b", r"\bembrace\b",
        r"\bstreamline\b", r"\bsupport\b", r"\btailor(?:ing)?\b",
        r"\btraining with\b", r"\bplan\b", r"\bprepare\b",
    ),
    "L300": (
        r"\banaly[sz]e\b", r"\bintegrate\b", r"\barchitect\b",
        r"\bdesign\b", r"\bimplement\b", r"\bdebug\b",
        r"\btroubleshoot\b", r"\borchestrate\b", r"\bextend\b",
        r"\bautomate\b", r"\binvestigate\b", r"\bmonitor\b",
    ),
    "L400": (
        r"\bevaluate\b", r"\boptimize\b", r"\bbalance\b", r"\bcompare\b",
        r"\bcontrast\b", r"\bjustify\b", r"\brecommend\b",
        r"\bprioritize\b", r"\bforecast\b", r"\bassess\b",
        r"\bmeasure\b", r"\bmitigate\b", r"\bselect\b",
        r"\bchoos(?:e|ing)\b", r"\bmaximize\b",
    ),
    "L500": (r"^scale\b", r"\boriginate\b", r"\breframe\b", r"\binfluence\b"),
}

def assign_rubric_level(title):
    normalized_title = title.strip().lower()
    evidence_by_level = {
        level: [
            match.group(0)
            for pattern in patterns
            if (match := re.search(pattern, normalized_title))
        ]
        for level, patterns in rubric_signals.items()
    }
    matched_levels = [level for level, evidence in evidence_by_level.items() if evidence]
    assigned_level = max(matched_levels, key=rubric_rank.get) if matched_levels else "L200"
    evidence = ", ".join(evidence_by_level[assigned_level]) or "general application (default)"
    return assigned_level, evidence

In [4]:
tagged_modules = []
for module in modules:
    rubric_level, rubric_evidence = assign_rubric_level(module["title"])
    tagged_modules.append({
        "uid": module["uid"],
        "title": module["title"],
        "learn_url": module["learn_url"],
        "catalog_level": module["levels"].strip().lower(),
        "rubric_level": rubric_level,
        "rubric_evidence": rubric_evidence,
    })

## Rubric-level distribution

In [5]:
rubric_counts = Counter(module["rubric_level"] for module in tagged_modules)
rubric_summary = [
    {"rubric_level": level, "module_count": rubric_counts[level]}
    for level in rubric_level_order
]
rubric_summary

[{'rubric_level': 'L100', 'module_count': 34},
 {'rubric_level': 'L200', 'module_count': 74},
 {'rubric_level': 'L300', 'module_count': 36},
 {'rubric_level': 'L400', 'module_count': 10},
 {'rubric_level': 'L500', 'module_count': 1}]

## Catalog level by rubric level

Counts show how the five rubric levels are distributed inside each existing three-level catalog label.

In [6]:
cross_tab = []
for catalog_level in catalog_level_order:
    row = {"catalog_level": catalog_level.title()}
    matching_modules = [
        module for module in tagged_modules if module["catalog_level"] == catalog_level
    ]
    row.update({
        rubric_level: sum(module["rubric_level"] == rubric_level for module in matching_modules)
        for rubric_level in rubric_level_order
    })
    cross_tab.append(row)
cross_tab

[{'catalog_level': 'Beginner', 'L100': 28, 'L200': 33, 'L300': 3, 'L400': 1, 'L500': 0},
 {'catalog_level': 'Intermediate', 'L100': 6, 'L200': 36, 'L300': 19, 'L400': 8, 'L500': 0},
 {'catalog_level': 'Advanced', 'L100': 0, 'L200': 5, 'L300': 14, 'L400': 1, 'L500': 1}]

In [7]:
cross_tab_percent = []
for row in cross_tab:
    total = sum(row[level] for level in rubric_level_order)
    cross_tab_percent.append({
        "catalog_level": row["catalog_level"],
        **{level: f"{row[level] / total:.1%}" for level in rubric_level_order},
    })
cross_tab_percent

[{'catalog_level': 'Beginner', 'L100': '43.1%', 'L200': '50.8%', 'L300': '4.6%', 'L400': '1.5%', 'L500': '0.0%'},
 {'catalog_level': 'Intermediate', 'L100': '8.7%', 'L200': '52.2%', 'L300': '27.5%', 'L400': '11.6%', 'L500': '0.0%'},
 {'catalog_level': 'Advanced', 'L100': '0.0%', 'L200': '23.8%', 'L300': '66.7%', 'L400': '4.8%', 'L500': '4.8%'}]

## Alignment

To compare systems with different numbers of categories, the rubric levels are condensed into three ordered bands: **L100 → Beginner**, **L200-L300 → Intermediate**, and **L400-L500 → Advanced**. Agreement is the share of identical bands. Cohen's kappa corrects that agreement for chance; linear-weighted kappa gives partial credit when labels differ by one adjacent band.

In [8]:
rubric_to_band = {
    "L100": "beginner",
    "L200": "intermediate",
    "L300": "intermediate",
    "L400": "advanced",
    "L500": "advanced",
}
band_order = ("beginner", "intermediate", "advanced")
band_rank = {level: rank for rank, level in enumerate(band_order)}
level_pairs = [
    (module["catalog_level"], rubric_to_band[module["rubric_level"]])
    for module in tagged_modules
]
module_count = len(level_pairs)

observed_agreement = sum(left == right for left, right in level_pairs) / module_count
catalog_band_counts = Counter(left for left, _ in level_pairs)
rubric_band_counts = Counter(right for _, right in level_pairs)
expected_agreement = sum(
    catalog_band_counts[level] * rubric_band_counts[level] for level in band_order
) / module_count**2
cohens_kappa = (observed_agreement - expected_agreement) / (1 - expected_agreement)

observed_weighted_disagreement = sum(
    abs(band_rank[left] - band_rank[right]) / (len(band_order) - 1)
    for left, right in level_pairs
) / module_count
expected_weighted_disagreement = sum(
    catalog_band_counts[left]
    * rubric_band_counts[right]
    * abs(band_rank[left] - band_rank[right])
    / (len(band_order) - 1)
    for left in band_order
    for right in band_order
) / module_count**2
weighted_kappa = 1 - observed_weighted_disagreement / expected_weighted_disagreement
mean_absolute_difference = sum(
    abs(band_rank[left] - band_rank[right]) for left, right in level_pairs
) / module_count

alignment_metrics = {
    "band_agreement": f"{observed_agreement:.1%}",
    "cohens_kappa": round(cohens_kappa, 3),
    "linear_weighted_kappa": round(weighted_kappa, 3),
    "mean_absolute_band_difference": round(mean_absolute_difference, 3),
}
alignment_metrics

{'band_agreement': '54.8%',
 'cohens_kappa': 0.225,
 'linear_weighted_kappa': 0.286,
 'mean_absolute_band_difference': 0.458}

**Interpretation:** The condensed labels match for **54.8%** of modules. Cohen's kappa is **0.225**, and linear-weighted kappa is **0.286**, indicating weak alignment beyond chance. The largest difference is in catalog-advanced modules: **66.7%** receive an L300 rubric tag, while only **9.6%** receive L400 or L500. Because this pass uses title signals rather than complete module content, the result should be treated as a hypothesis for human review, not a validity result for the rubric.

## Module-level tags

`tagged_modules` contains both labels and the matched title signal for every module. The next cell lists the disagreements first.

In [9]:
modules_by_disagreement = sorted(
    tagged_modules,
    key=lambda module: (
        rubric_to_band[module["rubric_level"]] == module["catalog_level"],
        module["catalog_level"],
        module["title"],
    ),
)
modules_by_disagreement

## Export the module-level comparison

`level_difference` compares the condensed rubric band with the catalog level. A value of **0** means aligned, a positive value means the rubric level is higher, and a negative value means the rubric level is lower. Filter this column to values other than zero to review misaligned modules.

In [10]:
alignment_rows = [
    {
        "module_title": module["title"],
        "module_url": module["learn_url"],
        "catalog_level": module["catalog_level"],
        "rubric_level": module["rubric_level"],
        "level_difference": (
            band_rank[rubric_to_band[module["rubric_level"]]]
            - band_rank[module["catalog_level"]]
        ),
    }
    for module in tagged_modules
]

alignment_csv_path = csv_path.with_name("microsoft-foundry-module-level-alignment.csv")
with alignment_csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=alignment_rows[0].keys())
    writer.writeheader()
    writer.writerows(alignment_rows)

alignment_csv_path

PosixPath('content-data/microsoft-foundry-module-level-alignment.csv')